# MODELO HIERÁRQUICO LINEAR DE DOIS NÍVEIS (HLM2)

O Modelo Hierárquico Linear (HLM) de dois níveis é usado quando os dados têm uma estrutura agrupada,
ou seja, quando os indivíduos (nível 1) estão aninhados dentro de grupos (nível 2).

Aqui está a explicação em 3 pontos simples:

   1. O Objetivo: Modelar como as variáveis no nível do indivíduo (ex: horas de estudo)
      e no nível do grupo (ex: escola) influenciam o resultado (desempenho).
   2. Os Dois Níveis: O nível 1 captura a variação entre indivíduos e o nível 2 captura
      a variação entre grupos (escolas), permitindo que cada grupo tenha sua própria interceptação.
   3. A Previsão: Com o modelo pronto, podemos prever o desempenho para novos alunos,
      considerando tanto o efeito individual quanto o efeito da escola.

   A fórmula básica que você verá no notebook é:
   Nível 1: Y_ij = β0j + β1j * X_ij + e_ij
   Nível 2: β0j = γ00 + u0j
   Juntos: Y_ij = γ00 + γ01 * W_j + β1j * X_ij + u0j + e_ij
    * Y_ij: Desempenho do aluno i na escola j.
    * X_ij: Variável de nível 1 (horas de estudo).
    * W_j: Variável de nível 2 (escola).
    * γ00: Intercepto global.
    * u0j: Efeito aleatório da escola.
    * e_ij: Erro do nível 1.

   Em resumo: é separar a variação entre alunos e entre escolas para entender como cada nível afeta o resultado.

---

### EXEMPLO 1 - Desempenho escolar agrupado por escola

1 - Importação dos pacotes

In [ ]:
import pandas as pd # manipulação de dados em formato de dataframe
import numpy as np # operações matemáticas
import seaborn as sns # visualização gráfica
import matplotlib.pyplot as plt # visualização gráfica
import statsmodels.api as sm # estimação de modelos
from scipy import stats # estatística chi2
from statsmodels.iolib.summary2 import summary_col # comparação entre modelos
from scipy.stats import gaussian_kde # inserção de KDEs em gráficos
from matplotlib.gridspec import GridSpec # plotagem de gráficos separados
import time # definição do intervalo de tempo entre gráficos com animação
import imageio # para geração de figura GIF
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM # estimação
# de modelos multinível logísticos
import plotly.graph_objs as go # gráfico 3D

import warnings
warnings.filterwarnings('ignore')

2 - Carregando Dados

In [ ]:
# CARREGAMENTO DA BASE DE DADOS
df_aluno_escola = pd.read_csv('desempenho_aluno_escola.csv', delimiter=',')

2.1 - Características das variáveis do dataset

In [ ]:
# Características das variáveis do dataset
df_aluno_escola.info()

2.2 - Estatísticas univariadas

In [ ]:
# Estatísticas univariadas
df_aluno_escola.describe()

2.3 - Categorização das variáveis 'estudante' e 'escola'

In [ ]:
# Atribuição de categorias para as variáveis 'estudante' e 'escola'
df_aluno_escola['estudante'] = df_aluno_escola['estudante'].astype('category')
df_aluno_escola['escola'] = df_aluno_escola['escola'].astype('category')

df_aluno_escola.info()

3 - Análise exploratória do dataset

In [ ]:
# Estudo sobre o desbalanceamento dos dados por escola
df_aluno_escola.groupby('escola')['estudante'].count().reset_index()

In [ ]:
# Desempenho médio dos estudantes por escola
desempenho_medio = df_aluno_escola.groupby('escola')['desempenho'].mean().reset_index()
desempenho_medio

In [ ]:
# Gráfico do desempenho escolar médio dos estudantes por escola

plt.figure(figsize=(15,10))
plt.plot(desempenho_medio['escola'], desempenho_medio['desempenho'],
         linewidth=5, color='indigo')
plt.scatter(df_aluno_escola['escola'], df_aluno_escola['desempenho'],
            alpha=0.5, color='orange', s=150)
plt.xlabel('Escola $j$ (nível 2)', fontsize=20)
plt.ylabel('Desempenho Escolar', fontsize=20)
plt.xticks(desempenho_medio.escola, fontsize=17)
plt.yticks(fontsize=17)
plt.show()

In [ ]:
# Boxplot da variável dependente ('desempenho')

plt.figure(figsize=(15,10))
sns.boxplot(data=df_aluno_escola, y='desempenho',
            linewidth=2, orient='v', color='deepskyblue')
sns.stripplot(data=df_aluno_escola, y='desempenho',
              color='darkorange', jitter=0.1, size=12, alpha=0.5)
plt.ylabel('Desempenho Escolar', fontsize=20)
plt.yticks(fontsize=17)
plt.show()

In [ ]:
# Kernel density estimation (KDE) - função densidade de probabilidade
# da variável dependente ('desempenho'), com histograma

plt.figure(figsize=(15,10))
sns.histplot(data=df_aluno_escola['desempenho'], kde=True,
             bins=30, color='deepskyblue')
plt.xlabel('Desempenho Escolar', fontsize=20)
plt.ylabel('Contagem', fontsize=20)
plt.tick_params(axis='y', labelsize=17)
plt.tick_params(axis='x', labelsize=17)
plt.show()

In [ ]:
# Boxplot da variável dependente ('desempenho') por escola

plt.figure(figsize=(15,10))
sns.boxplot(data=df_aluno_escola, x='escola', y='desempenho',
            linewidth=2, orient='v', palette='viridis')
sns.stripplot(data=df_aluno_escola, x='escola', y='desempenho',
              palette='viridis', jitter=0.2, size=8, alpha=0.5)
plt.ylabel('Desempenho Escolar', fontsize=20)
plt.xlabel('Escola $j$ (nível 2)', fontsize=20)
plt.tick_params(axis='y', labelsize=17)
plt.tick_params(axis='x', labelsize=17)
plt.show()

In [ ]:
# Kernel density estimation (KDE) - função densidade de probabilidade
# da variável dependente ('desempenho') por escola

escolas = df_aluno_escola['escola'].unique()
colors = sns.color_palette('viridis', len(escolas))

plt.figure(figsize=(15, 10))
g = sns.pairplot(df_aluno_escola[['escola', 'desempenho']], hue='escola',
                 height=8,
                 aspect=1.5, palette=colors)
g._legend.remove()
g.set(xlabel=None)
g.set(ylabel=None)
g.tick_params(axis='both', which='major', labelsize=15)

# Gera a legenda com cores e rótulos das escolas
legend_elements = [plt.Line2D([0], [0], marker='o', color='w',
                              markerfacecolor=color,
                              markersize=10, label=escola)
                   for escola, color in zip(escolas, colors)]
plt.legend(handles=legend_elements, title='Escola', fontsize=14,
           title_fontsize=18)

# Adiciona os rótulos diretamente na figura
plt.gcf().text(0.5, -0.01, 'Desempenho Escolar', ha='center', fontsize=20)
plt.gcf().text(-0.01, 0.5, 'Frequência', va='center', rotation='vertical',
               fontsize=20)
plt.show()

In [ ]:
# Kernel density estimation (KDE) - função densidade de probabilidade
# da variável dependente ('desempenho'), com histograma e por escola separadamente
# (função 'GridSpec' do pacote 'matplotlib.gridspec')

escolas = df_aluno_escola['escola'].unique()

fig = plt.figure(figsize=(15, 14))
gs = GridSpec(len(escolas) // 2 + 1, 2, figure=fig)

for i, escola in enumerate(escolas):
    ax = fig.add_subplot(gs[i])

    # Subset dos dados por escola
    df_escola = df_aluno_escola[df_aluno_escola['escola'] == escola]

    # Densidade dos dados
    densidade = gaussian_kde(df_escola['desempenho'])
    x_vals = np.linspace(min(df_escola['desempenho']),
                         max(df_escola['desempenho']), len(df_escola))
    y_vals = densidade(x_vals)

    # Plotagem da density area
    ax.fill_between(x_vals, y_vals,
                    color=sns.color_palette('viridis',
                                            as_cmap=True)(i/len(escolas)),
                    alpha=0.3)
    
    # Adiciona o histograma
    sns.histplot(df_escola['desempenho'], ax=ax, stat='density', color='black',
                 edgecolor='black', fill=True,
                 bins=15, alpha=0.1)
    ax.set_title(f'Escola {escola}', fontsize=15)
    ax.set_ylabel('Densidade')
    ax.set_xlabel('Desempenho')

plt.tight_layout()
plt.show()

4 - Visualização da relação entre horas e desempenho

In [ ]:
# Gráfico de desempenho x horas (OLS)

plt.figure(figsize=(15,10))
sns.regplot(data=df_aluno_escola, x='horas', y='desempenho', marker='o', ci=False,
            scatter_kws={'color':'dodgerblue', 'alpha':0.8, 's':200},
            line_kws={'color':'grey', 'linewidth': 5})
plt.xlabel('Quantidade Semanal de Horas de Estudo do Aluno', fontsize=20)
plt.ylabel('Desempenho Escolar', fontsize=20)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.show()

In [ ]:
# Gráfico de desempenho x horas (OLS) por escola separadamente
# Animação no ambiente Plots

# Obtenção da lista de escolas
escolas = df_aluno_escola['escola'].unique()

# Definição do número de cores na paleta viridis
num_cores = len(escolas)

# Criação do dicionário de mapeamento da escola -> cor
cor_escola = dict(zip(escolas, sns.color_palette('viridis', num_cores)))

while True:
    # Loop para cada escola
    for escola in escolas:
        # Filtro dos dados para determinada escola
        data = df_aluno_escola[df_aluno_escola['escola'] == escola]

        # Criação do lmplot com a cor específica
        sns.lmplot(x='horas', y='desempenho', data=data, hue='escola',
                   height=6, aspect=1.5, ci=False, palette=[cor_escola[escola]])
        plt.title(f'Desempenho Escolar - Escola {escola}', fontsize=20)
        plt.xlabel('Quantidade Semanal de Horas de Estudo do Aluno', fontsize=20)
        plt.ylabel('Desempenho Escolar', fontsize=20)
        plt.yticks(np.arange(0, 101, 20), fontsize=14)
        plt.xticks(np.arange(0, 36, 5), fontsize=14)
        plt.tight_layout()

        # Plotagem da figura
        plt.show()

        # Intervalo de tempo entre os gráficos
        time.sleep(1)

In [ ]:
# Gráfico de desempenho x horas (OLS) por escola separadamente
# Geração de uma Figura GIF

# Obtenção da lista de escolas
escolas = df_aluno_escola['escola'].unique()

# Definição do número de cores na paleta viridis
num_cores = len(escolas)

# Criação do dicionário de mapeamento da escola -> cor
cor_escola = dict(zip(escolas, sns.color_palette('viridis', num_cores)))

# Lista para armazenar os frames dos gráficos
frames = []

# Loop para cada escola
for escola in escolas:
    # Filtro dos dados para determinada escola
    data = df_aluno_escola[df_aluno_escola['escola'] == escola]

    # Criação do lmplot com a cor específica
    sns.lmplot(x='horas', y='desempenho', data=data, hue='escola',
               height=6, aspect=1.5, ci=False, palette=[cor_escola[escola]])
    plt.title(f'Desempenho escolar - Escola {escola}')
    plt.xlabel('Quantidade Semanal de Horas de Estudo do Aluno')
    plt.ylabel('Desempenho Escolar')
    plt.yticks(np.arange(0, 101, 20))
    plt.xticks(np.arange(0, 36, 5))
    plt.tight_layout()
    
    # Converte o gráfico em um array de imagens (compatível com Matplotlib novo)
    plt_canvas = plt.get_current_fig_manager().canvas
    plt_canvas.draw()

    image = np.asarray(plt_canvas.buffer_rgba())  # RGBA
    image = image[:, :, :3]  # converte para RGB

    # Anexa o array de imagens à lista de quadros (frames)
    frames.append(image)

    # Limpa o gráfico para a próxima iteração
    plt.close()

# Salva os quadros (frames) como um GIF
imageio.mimsave('graficos_escolas.gif', frames, fps=1)

# Mostra o GIF
plt.imshow(frames[0])
plt.axis('off')
plt.show()

In [ ]:
# Gráfico de desempenho escolar em função da variável 'horas'
# Variação entre estudantes de uma mesma escola e entre escolas diferentes
# Visualização do contexto!
# NOTE QUE A PERSPECTIVA MULTINÍVEL NATURALMENTE CONSIDERA O COMPORTAMENTO
# HETEROCEDÁSTICO NOS DADOS!

palette = sns.color_palette('viridis',
                            len(df_aluno_escola['escola'].unique()))

plt.figure(figsize=(15,10))
sns.scatterplot(data=df_aluno_escola, x='horas', y='desempenho', hue='escola',
                palette=palette, s=200, alpha=0.8, edgecolor='w')

for escola in df_aluno_escola['escola'].cat.categories:
    subset = df_aluno_escola[df_aluno_escola['escola'] == escola]
    sns.regplot(data=subset, x='horas', y='desempenho', scatter=False, ci=False,
                line_kws={'color': palette[df_aluno_escola['escola'].cat.categories.get_loc(escola)], 'linewidth': 5})

plt.xlabel('Quantidade Semanal de Horas de Estudo do Aluno', fontsize=20)
plt.ylabel('Desempenho Escolar', fontsize=20)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(title='Escola', title_fontsize='14', fontsize='13', loc='upper left')
plt.show()

5 - Estimação do Modelo Nulo (HLM2 - Dois Níveis)

In [ ]:
# Estimação do modelo nulo (função 'MixedLM' do pacote 'statsmodels')

modelo_nulo_hlm2 = sm.MixedLM.from_formula(formula='desempenho ~ 1',
                                            groups='escola',
                                            re_formula='1',
                                            data=df_aluno_escola).fit()

# Parâmetros do 'modelo_nulo_hlm2'
modelo_nulo_hlm2.summary()

In [ ]:
# In[1.14]: Análise da significância estatística dos efeitos aleatórios de intercepto

teste = float(modelo_nulo_hlm2.cov_re.iloc[0, 0]) / \
    float(pd.DataFrame(modelo_nulo_hlm2.summary().tables[1]).iloc[1, 1])

p_value = 2 * (1 - stats.norm.cdf(abs(teste)))

print(f"Estatística z para a Significância dos Efeitos Aleatórios: {teste:.3f}")
print(f"P-valor: {p_value:.3f}")

if p_value >= 0.05:
    print("Ausência de significância estatística dos efeitos aleatórios ao nível de confiança de 95%.")
else:
    print("Efeitos aleatórios contextuais significantes ao nível de confiança de 95%.")

# Análise do Modelo Hierárquico Linear de Dois Níveis (HLM2)

## Resumo do Modelo

O objetivo deste modelo é analisar o desempenho escolar dos alunos, considerando a estrutura
hierárquica dos dados onde os estudantes (nível 1) estão aninhados dentro de escolas (nível 2).
O modelo nulo (sem variáveis explicativas) permite verificar se há variação significativa
entre as escolas.

### Equação do Modelo Hierárquico

[
Nível 1: Y_ij = β0j + e_ij
]
[
Nível 2: β0j = γ00 + u0j
]

Juntos:
[
Y_ij = γ00 + u0j + e_ij
]

Onde:

* **γ00:** Intercepto global (média geral de desempenho).
* **u0j:** Efeito aleatório da escola j (variação entre escolas).
* **e_ij:** Erro do nível 1 (variação entre alunos dentro da mesma escola).

---

# Qualidade do Ajuste

# Resumo: Modelo Hierárquico Linear de Dois Níveis (HLM2)

Este documento apresenta um resumo do fluxo de código do notebook, focado na análise do
**desempenho escolar** agrupado por **escola**.

---

## 1. Preparação e Exploração
- **Objetivo:** Entender como as variáveis de nível 1 (horas de estudo) e nível 2 (escola)
  influenciam o desempenho escolar.
- **Dados:** O arquivo `desempenho_aluno_escola.csv` é carregado.
- **Análise Inicial:** Uso de `describe()` para estatísticas descritivas e `info()` para verificar tipos de dados.
- **Categorização:** Variáveis `estudante` e `escola` são convertidas para tipo category.

## 2. Visualização Inicial
- São gerados gráficos de dispersão, boxplots e KDEs para entender a distribuição do desempenho.
- Gráficos por escola permitem visualizar a variação entre grupos.

## 3. Construção do Modelo Nulo (HLM2)
O modelo de **Mínimos Quadrados Generalizados (MixedLM)** é estimado:
```python
modelo_nulo_hlm2 = sm.MixedLM.from_formula('desempenho ~ 1', groups='escola', re_formula='1', data=df_aluno_escola).fit()
```
- A fórmula `'desempenho ~ 1'` indica um modelo apenas com intercepto (modelo nulo).
- `groups='escola'` especifica o agrupamento no nível 2.
- `re_formula='1'` especifica efeito aleatório de intercepto.

## 4. Interpretação dos Resultados
Ao rodar `modelo_nulo_hlm2.summary()`, obtemos:
- **Intercepto (γ00):** Média geral de desempenho.
- **Variância entre escolas (u0j):** Indica se há diferenças significativas entre escolas.
- **Variância residual (e_ij):** Indica a variação dentro das escolas.
- **Teste de significância dos efeitos aleatórios:** Verifica se a variação entre escolas é significativa.

## 5. Conclusão
O modelo hierárquico permite separar a variação entre alunos e entre escolas, sendo essencial
para entender fatores contextuais e individuais que influenciam o desempenho escolar.

---
**Conclusão do Notebook:** O modelo hierárquico linear de dois níveis é utilizado para analisar
o desempenho escolar considerando a estrutura agrupada dos dados (alunos dentro de escolas).